# MarketReactionLab | V20 frozen-model second-look audit

**Reproducible Kaggle research notebook** · 2025–2026 stock sessions · CPU only

**Question.** Does the frozen, market-context LightGBM (V13) outperform the frozen price-only XGBoost (V9) on the *same* 2025–2026 stock sessions?

**Protocol.** The stored cohort contains 60,480 stock sessions across 448 dates. Models were trained before the evaluation window. **This is not an untouched holdout**: V10 previously opened the same calendar period. Price-only V13 does not demonstrate the value of financial headlines.

This notebook either audits saved `second_look_predictions.csv` or uses archived pre-2025 models to create that file. It does **not** retrain a model, change thresholds, or tune weights on 2025–2026.

**Sections:** 1. setup → 2. evidence loading → 3. record checks → 4. headline metrics → 5. clustered bootstrap → 6. plots → 7. export and limitations.

**Kaggle setup:** Internet ON if models must be downloaded, accelerator OFF. No API keys need to be printed.

## 1 · Environment and output directory

Install packages only if Kaggle lacks them. A saved predictions file allows quick offline auditing; otherwise the notebook downloads the archived frozen models through the Kaggle CLI.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    log_loss, roc_auc_score, precision_recall_fscore_support
)

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT = WORK / "marketreactionlab_v20_notebook"
OUTPUT.mkdir(parents=True, exist_ok=True)
LABELS = [0, 1, 2]
CLASS_NAMES = ["Negative", "Neutral", "Positive"]
print("Output:", OUTPUT)

## 2 · Obtain saved predictions (preferred) or recompute with frozen models

If a `second_look_predictions.csv` file is attached as a Kaggle dataset, this notebook reads it directly. Otherwise, it downloads the **V9 pre-2025** and **V13 pre-2025** artifacts and rebuilds the frozen predictions. Never substitute a 2026 **production** checkpoint, since such models were refit using the evaluation period.

In [ ]:
candidate_paths = [
    WORK / "marketreactionlab_v20_frozen_second_look" / "second_look_predictions.csv",
    WORK / "second_look_predictions.csv",
    Path.cwd() / "research" / "artifacts" / "v20" / "second_look_predictions.csv",
]
if Path("/kaggle/input").exists():
    candidate_paths.extend(Path("/kaggle/input").rglob("second_look_predictions.csv"))

pred_file = next((p for p in candidate_paths if p.is_file()), None)

if pred_file is None:
    import joblib
    P9 = WORK / "marketreactionlab_v9_2026"
    P13 = WORK / "marketreactionlab_v13_context"

    def restore(folder, identifier, required):
        folder.mkdir(parents=True, exist_ok=True)
        if not all((folder / f).exists() for f in required):
            cli = shutil.which("kaggle")
            if not cli:
                raise RuntimeError("Kaggle CLI unavailable. Attach the saved predictions CSV instead.")
            subprocess.run(
                [cli, "datasets", "download", "-d", identifier,
                 "-p", str(folder), "--unzip"], check=True
            )
        missing = [f for f in required if not (folder / f).exists()]
        if missing:
            raise FileNotFoundError(f"{folder}: missing {missing}")

    restore(P9, "abhijeetnayak63/marketreactionlab-v9-prices-2026",
            ["report.json", "v9_price_only_xgboost.joblib", "sealed_2025_2026.parquet"])
    restore(P13, "abhijeetnayak63/marketreactionlab-v13-context-20261019",
            ["v13_model_pre2025.joblib", "report.json"])

    base = json.loads((P9 / "report.json").read_text())
    source = pd.read_parquet(P9 / "sealed_2025_2026.parquet")
    source["date"] = pd.to_datetime(source["date"]).dt.normalize()
    source["target_date"] = pd.to_datetime(source["target_date"]).dt.normalize()
    assert len(source) == 60480
    assert source["session_id"].is_unique
    assert (source["target_date"] > source["date"]).all()

    y = source["reaction_label"].map({-1: 0, 0: 1, 1: 2})
    assert y.notna().all()
    x9 = source[base["features"]].copy()
    x9["feat_rolling_avg_vol_20d"] = np.log1p(
        x9["feat_rolling_avg_vol_20d"].clip(lower=0)
    )
    assert x9.shape[1] == 16

    market_fields = [
        "feat_ret_1d", "feat_ret_5d", "feat_ret_20d",
        "feat_rolling_vol_20d", "feat_rsi_14", "feat_relative_volume",
    ]
    medians = source.groupby("date")[market_fields].median()
    matched = medians.reindex(source["date"].to_numpy()).reset_index(drop=True)
    assert matched.notna().all().all()
    x13 = x9.reset_index(drop=True).copy()
    for field in market_fields:
        x13["market_" + field[5:]] = matched[field].to_numpy()
    for days in (1, 5, 20):
        x13[f"relative_return_{days}d"] = (
            x13[f"feat_ret_{days}d"] - x13[f"market_ret_{days}d"]
        )
    x13["relative_volatility"] = (
        x13["feat_rolling_vol_20d"] / (x13["market_rolling_vol_20d"] + 0.003)
    )
    x13["relative_rsi"] = x13["feat_rsi_14"] - x13["market_rsi_14"]
    x13["volume_surprise"] = np.log1p(x13["feat_relative_volume"].clip(lower=0))
    x13["vol_adjusted_momentum"] = (
        x13["relative_return_5d"] / (x13["feat_rolling_vol_20d"] + 0.005)
    )
    x13["market_direction"] = np.sign(x13["market_ret_5d"])
    assert x13.shape[1] == 30

    m9 = joblib.load(P9 / "v9_price_only_xgboost.joblib")
    m13 = joblib.load(P13 / "v13_model_pre2025.joblib")
    if hasattr(m13, "feature_name_"):
        assert list(m13.feature_name_) == list(x13.columns)
    a9 = np.asarray(m9.predict_proba(x9.to_numpy(np.float32)), float)
    a13 = np.asarray(m13.predict_proba(x13.astype("float32")), float)
    p9 = np.asarray(m9.predict(x9.to_numpy(np.float32)), int).ravel()
    p13 = np.asarray(m13.predict(x13.astype("float32")), int).ravel()

    paired = pd.DataFrame({
        "session_id": source.session_id.to_numpy(), "date": source.date.to_numpy(),
        "actual": y.to_numpy(int), "v9": p9, "v13": p13,
        "v9_prob_negative": a9[:, 0], "v9_prob_neutral": a9[:, 1],
        "v9_prob_positive": a9[:, 2],
        "v13_prob_negative": a13[:, 0], "v13_prob_neutral": a13[:, 1],
        "v13_prob_positive": a13[:, 2],
    })
    pred_file = OUTPUT / "second_look_predictions.csv"
    paired.to_csv(pred_file, index=False)
    print("Created frozen-model predictions:", pred_file)
else:
    print("Using saved paired predictions:", pred_file)

paired = pd.read_csv(pred_file)
paired["date"] = pd.to_datetime(paired["date"]).dt.normalize()
paired["week"] = paired["date"].dt.to_period("W-FRI").astype(str)
print("Observations:", len(paired), "| trading dates:", paired.date.nunique())

## 3 · Validate record alignment and probability semantics

The models must predict the same unique sessions, using class order **negative / neutral / positive**. An earlier scikit-learn warning about tiny probability-sum rounding errors does not by itself prove bad predictions; we explicitly check ordering and normalise only for numerical scoring.

In [ ]:
required = [
    "session_id", "date", "actual", "v9", "v13",
    "v9_prob_negative", "v9_prob_neutral", "v9_prob_positive",
    "v13_prob_negative", "v13_prob_neutral", "v13_prob_positive"
]
assert set(required).issubset(paired.columns)
assert len(paired) == 60480
assert paired.session_id.is_unique
assert paired.date.nunique() == 448

y = paired.actual.to_numpy(int)
p9 = paired.v9.to_numpy(int)
p13 = paired.v13.to_numpy(int)
assert set(np.unique(np.concatenate([y, p9, p13]))).issubset({0, 1, 2})

def checked_probs(prefix, predictions):
    probs = paired[[f"{prefix}_prob_{c}" for c in ("negative", "neutral", "positive")]].to_numpy(float)
    assert probs.shape == (len(paired), 3)
    assert np.isfinite(probs).all()
    assert (probs >= 0).all()
    assert np.max(np.abs(probs.sum(axis=1) - 1)) < 0.002
    assert np.array_equal(probs.argmax(axis=1), predictions)
    probs = np.maximum(probs, np.finfo(float).eps)
    return probs / probs.sum(axis=1, keepdims=True)

a9 = checked_probs("v9", p9)
a13 = checked_probs("v13", p13)
assert paired.date.min() >= pd.Timestamp("2025-01-01")
print("PASS: unique session keys, label order, class argmax, finite probabilities.")

## 4 · Primary and secondary metrics

**Macro-F1** is the predefined headline metric; accuracy, log-loss and one-versus-rest ROC-AUC describe different aspects of behaviour. Big-move F1 and conditional sign accuracy distinguish volatility/tail detection from directional skill.

In [ ]:
def row(name, prediction, probabilities):
    mask = (y != 1) & (prediction != 1)
    return {
        "Model": name,
        "Macro-F1": f1_score(y, prediction, average="macro", labels=LABELS, zero_division=0),
        "Accuracy": accuracy_score(y, prediction),
        "Log-loss": log_loss(y, probabilities, labels=LABELS),
        "Macro OVR AUC": roc_auc_score(y, probabilities, labels=LABELS, multi_class="ovr", average="macro"),
        "Big-move F1": f1_score(y != 1, prediction != 1, zero_division=0),
        "Sign accuracy on predicted tails": float(np.mean(y[mask] == prediction[mask])) if mask.any() else np.nan,
    }

summary = pd.DataFrame([row("V9: price XGBoost", p9, a9), row("V13: context LightGBM", p13, a13)])
display(summary.style.format(precision=6))
only13 = int(((p13 == y) & (p9 != y)).sum())
only9 = int(((p9 == y) & (p13 != y)).sum())
delta = float(summary.iloc[1]["Macro-F1"] - summary.iloc[0]["Macro-F1"])
print(f"Macro-F1 difference (V13−V9): {delta:+.6f}")
print(f"V13 only correct: {only13:,} | V9 only correct: {only9:,} | net: {only13-only9:+,}")

## 5 · Paired date/week bootstrap (10,000 iterations)

We resample **groups of trading dates**, not individual stock predictions, because stocks share market shocks. The week-block variant partially respects within-week dependence; neither interval includes uncertainty from trying many prior model configurations or selecting on previously inspected periods.

In [ ]:
def f1_from_cm(cm):
    cm = np.asarray(cm, float)
    tp = np.diag(cm)
    denom = cm.sum(axis=0) + cm.sum(axis=1)
    return float(np.divide(2 * tp, denom, out=np.zeros(3), where=denom > 0).mean())

def paired_bootstrap(group_name, n=10000, seed=20261021):
    groups = list(paired.groupby(group_name).indices.values())
    left = np.stack([confusion_matrix(y[ix], p9[ix], labels=LABELS) for ix in groups])
    right = np.stack([confusion_matrix(y[ix], p13[ix], labels=LABELS) for ix in groups])
    rng = np.random.default_rng(seed)
    diffs = np.empty(n)
    for b in range(n):
        ids = rng.integers(0, len(groups), len(groups))
        diffs[b] = f1_from_cm(right[ids].sum(axis=0)) - f1_from_cm(left[ids].sum(axis=0))
    return np.quantile(diffs, [0.025, 0.975]).tolist()

date_ci = paired_bootstrap("date")
week_ci = paired_bootstrap("week")
print("95% date-cluster bootstrap CI:", [round(x, 6) for x in date_ci])
print("95% week-block bootstrap CI:", [round(x, 6) for x in week_ci])
print("Intervals are exploratory: 2025–2026 was previously examined.")

## 6 · Research figures from individual saved predictions

These are **computed figures**, not screenshot reconstructions. Labels and sample counts are generated directly from the 60,480-row CSV. Every model comparison remains within this 2025–2026 cohort.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.3))
v = summary["Macro-F1"].to_numpy()
bars = ax.bar(summary.Model, v)
ax.set_ylim(0, max(v) * 1.2)
ax.set_ylabel("Macro-F1")
ax.set_title("Frozen-model second look · 2025–2026")
for bar, val in zip(bars, v):
    ax.text(bar.get_x() + bar.get_width()/2, val + .004, f"{val:.4f}", ha="center")
fig.tight_layout()
fig.savefig(OUTPUT / "v20_macro_f1.png", dpi=180)
plt.show()

for name, pred in (("V9 price XGBoost", p9), ("V13 context LightGBM", p13)):
    cm = confusion_matrix(y, pred, labels=LABELS)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    image = ax.imshow(cm)
    ax.set_xticks(range(3), CLASS_NAMES)
    ax.set_yticks(range(3), CLASS_NAMES)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
    ax.set_title(name + " · 2025–2026")
    for (i, j), value in np.ndenumerate(cm):
        ax.text(j, i, f"{value:,}", ha="center", va="center",
                color="white" if value > cm.max()/2 else "black")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(OUTPUT / ("v20_" + name.split()[0].lower() + "_confusion.png"), dpi=180)
    plt.show()

## 7 · Year-by-year descriptive check and export

Separating 2025 and 2026 is a robustness description, **not** a new held-out test. The original sample and model-fitting decisions were already influenced by prior analyses.

In [ ]:
annual = []
for year, d in paired.groupby(paired.date.dt.year, sort=True):
    yy = d.actual.to_numpy(int)
    f9 = f1_score(yy, d.v9, average="macro", labels=LABELS, zero_division=0)
    f13 = f1_score(yy, d.v13, average="macro", labels=LABELS, zero_division=0)
    annual.append({"year": int(year), "n": len(d), "v9_macro_f1": float(f9),
                   "v13_macro_f1": float(f13), "difference": float(f13-f9)})
annual = pd.DataFrame(annual)
display(annual)

result = {
    "status": "post-hoc frozen-model second-look, NOT untouched",
    "samples": len(paired),
    "trading_dates": paired.date.nunique(),
    "macro_f1_delta": delta,
    "date_bootstrap_95pct": date_ci,
    "week_bootstrap_95pct": week_ci,
    "v13_only_correct": only13,
    "v9_only_correct": only9,
    "per_model": summary.set_index("Model").to_dict(orient="index"),
    "annual": annual.to_dict(orient="records"),
    "limitations": [
        "No new training was performed by this notebook",
        "2025–2026 was previously examined in V10",
        "Algorithms and features differ; this is not a controlled feature ablation",
        "Cross-stock medians require a full same-session ticker panel",
        "Directional sign accuracy is near chance; no trading-profit claim",
    ],
}
(OUTPUT / "v20_notebook_audit.json").write_text(json.dumps(result, indent=2))
annual.to_csv(OUTPUT / "v20_annual_metrics.csv", index=False)
summary.to_csv(OUTPUT / "v20_model_metrics.csv", index=False)
print("Research outputs saved in", OUTPUT)

---

### Interpretation & responsible reporting

- This is a **price-only** experiment using the historical frozen V9 and V13 pipelines, not evidence that news or tool agents improve forecasting.
- The 2025–2026 period is a **second look**; confidence intervals are descriptive and cannot remove selection optimism.
- Macro-F1 improvement may reflect better **non-neutral movement detection** more than accurate positive-versus-negative signs.
- Different algorithms and additional context features changed together, so the comparison is **not a causal feature ablation**.
- The historical headline dataset used in other cohorts has **untrusted date-only timestamps**; results must never be presented as verified point-in-time prediction.
- Full future-performance and trading-profit claims require a genuinely untouched chronological evaluation and complete provenance audit.

**Source:** MarketReactionLab project data and frozen V9/V13 Kaggle archives; paper refers to this run as **V20**.